# 04 - Modelado y pronóstico del consumo eléctrico

En este cuaderno se entrena y evalúa un modelo de regresión para predecir la potencia activa global a nivel horario utilizando las características temporales generadas previamente.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
sns.set(style="whitegrid")

## Carga de datos enriquecidos para modelado

In [5]:
features_path = Path("..") / "data" / "household_power_consumption.csv"
df = pd.read_csv(features_path)

df.head()

C:\Users\frany\AppData\Local\Temp\ipykernel_3680\3055846037.py:2: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(features_path)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [7]:
df.columns

Index(['Date', 'Time', 'Global_active_power', 'Global_reactive_power',
       'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3'],
      dtype='object')

## Definición de variables de entrada y objetivo

In [ ]:

gap_col = [c for c in df.columns if c.lower() == "global_active_power"][0]

feature_cols = [
    "hour",
    "day_of_week",
    "is_weekend",
    "lag_1",
    "lag_24",
    "rolling_mean_24",
]

X = df[feature_cols].copy()
y = df[gap_col].copy()

X.shape, y.shape

KeyError: "None of [Index(['hour', 'day_of_week', 'is_weekend', 'lag_1', 'lag_24',\n       'rolling_mean_24'],\n      dtype='object')] are in the [columns]"

## División entrenamiento / prueba respetando el orden temporal

In [ ]:
split_idx = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

X_train.shape, X_test.shape

## Entrenamiento de un modelo Random Forest Regressor

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

## Evaluación del modelo en el conjunto de prueba

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

mae, rmse, r2

## Comparación visual entre valores reales y predichos

In [ ]:
results = pd.DataFrame({"y_test": y_test, "y_pred": y_pred}, index=y_test.index)

sample = results.iloc[:24 * 7]

plt.figure(figsize=(12, 5))
sample["y_test"].plot(label="Real")
sample["y_pred"].plot(label="Predicho")
plt.ylabel(gap_col)
plt.title("Consumo real vs predicho en la primera semana del conjunto de prueba")
plt.legend()
plt.tight_layout()

## Importancia de características

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=importances.values, y=importances.index)
plt.title("Importancia de características en el modelo Random Forest")
plt.xlabel("Importancia")
plt.tight_layout()